# Calculate the Label

I will simulate a cost of a visit (charged cost) using the features included.

**Calculation Method**

- Draw costs based on ICD/CCSR codes from Log-Normal Dist
  - cost = icd10_cm_code + ccsr_code 
  - Draw each feature from log-normal distributions with paramters determined by codes
  - Incorporates randomness
- Add base equation
  - cost += AWEEKEND + ELECTIVE + (AWEEKEND\*ELECTIVE) + AGE + LOS
- Adjustments:
  - Add \$2,000 to cost for age <= 2
  - Multiply the cost by 1.05 when age >= 65
  - Multiply the cost by 1.02 when FEMALE=1
  - Multiply the cost by 1.1 when HOSP_CONTROL_For_profit = 1
  - Subtract \$8,000 from the cost when HOSP_CONTROL_Public = 1
- Threshold to min cost

In [1]:
# Modify the file 00_setup.py to define input/output file paths on your system
from importlib.machinery import SourceFileLoader
setup = SourceFileLoader("setup", "./00_setup.py").load_module()

In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [3]:
import seaborn as sns

In [4]:
import matplotlib.pyplot as plt

In [5]:
from sklearn.model_selection import train_test_split

In [23]:
import numpy as np
from scipy import stats

In [35]:
rng = np.random.default_rng(34543)

## Data Import

#### Visits

In [18]:
visits_df = pd.read_parquet(Path(setup.output_path).joinpath('01_DATA_visits_features.parquet'))

#### Summary Stats 
This will guide the label creation

In [20]:
sum_stats =  pd.read_csv(Path(setup.output_path).joinpath('01_sum_stats.csv'))
sum_stats[sum_stats['var'] == 'TOTCHG_2023']

,Variable / Label,N,Minimum,Maximum,Mean,Median,Sum,var
57,TOTCHG_2023: Total charges adjusted for nation...,6700121,103,11278286,76295.9,40621,511191853831,TOTCHG_2023


In [22]:
cost_median = sum_stats[sum_stats['var'] == 'TOTCHG_2023']['Median'].iloc[0]
cost_mean = sum_stats[sum_stats['var'] == 'TOTCHG_2023']['Mean'].iloc[0]
cost_min = sum_stats[sum_stats['var'] == 'TOTCHG_2023']['Minimum'].iloc[0]
print(f"The mean cost is ${cost_mean:,.0f} and the median ${cost_median:,.0f}")

The mean cost is $76,296 and the median $40,621


#### ICD / CCSR Keys

In [52]:
feat_info_ccsr = pd.read_csv(Path(setup.output_path).joinpath('01_feat_ccsr_based_info.csv'))
feat_info_icd10 = pd.read_csv(Path(setup.output_path).joinpath('01_feat_icd_based_info.csv'))

In [73]:
icd10_map = pd.read_csv(Path(setup.output_path).joinpath('REF_icd_cm_code.csv'))

## Log Normal Function

In [27]:
def get_log_norm_dist(mean=cost_mean, median=cost_median):
    mu = np.log(median)
    sigma = np.sqrt(2 * np.log(mean / median))
    lognorm_dist = stats.lognorm(s=sigma, scale=np.exp(mu))
    return lognorm_dist

In [28]:
lnd = get_log_norm_dist()

In [29]:
print(f"Model Mean:        {lnd.mean():,.2f}")
print(f"Model Median:      {lnd.median():,.2f}")
print(f"Model Implied Std: {lnd.std():,.2f}")

Model Mean:        76,295.90
Model Median:      40,621.00
Model Implied Std: 121,302.74


## ICD10 and CCSR Parameters

Attribute about half of costs to the ICD10 code and to the CCSR code - so the average mean and median are as given above.

Use a log-normal distribution for the means and medians, determined by code

The ICD10 distributions are a bit wider than the CCSR ones

In [60]:
feat_info_ccsr['ln_mean_ccsr'] = pd.Series(rng.normal(loc=cost_mean/2, 
                                                      scale=cost_mean/8, 
                                                      size= len(feat_info_ccsr))) \
    .clip(cost_mean/100, None)

In [61]:
feat_info_ccsr['ln_median_ccsr'] = pd.Series(rng.normal(loc=cost_median/2, 
                                                      scale=cost_median/8, 
                                                      size= len(feat_info_ccsr))) \
    .clip(cost_median/100, None)

In [62]:
feat_info_ccsr.sample(5)

,ccsr_code,visits,female_rate,los_dist_split,ln_mean_ccsr,ln_median_ccsr
96,EYE006,1774.0,0.446370,0.990000,35180.999184,24577.146535
192,INJ061,48.0,0.774711,0.026230,47090.568897,18215.841410
25,CIR016,15774.0,0.610437,0.833433,47185.373615,18874.349139
122,GEN011,299.0,0.474762,0.990000,25526.512342,32176.649782
75,EAR003,4396.0,0.721652,0.990000,43123.011441,26776.007993


In [67]:
feat_info_ccsr[['ln_mean_ccsr', 'ln_median_ccsr']].describe()

,ln_mean_ccsr,ln_median_ccsr
count,441.000000,441.000000
mean,38572.022037,20584.073573
std,9379.335638,4980.398383
min,7387.715096,5065.427558
25%,32391.636429,17409.744950
50%,39050.098011,20394.596536
75%,44853.660340,24010.697381
max,65467.306876,35801.807476


In [63]:
feat_info_ccsr.to_csv(Path(setup.output_path).joinpath('02_feat_ccsr_based_info.csv'), index=False)

In [68]:
feat_info_icd10['ln_mean_icd10'] = pd.Series(rng.normal(loc=cost_mean/2, 
                                                      scale=cost_mean/4, 
                                                      size= len(feat_info_icd10))) \
    .clip(cost_mean/100, None)

In [69]:
feat_info_icd10['ln_median_icd10'] = pd.Series(rng.normal(loc=cost_median/2, 
                                                      scale=cost_median/4, 
                                                      size= len(feat_info_icd10))) \
    .clip(cost_median/100, None)

In [70]:
feat_info_icd10[['ln_mean_icd10', 'ln_median_icd10']].describe()

,ln_mean_icd10,ln_median_icd10
count,9103.000000,9103.000000
mean,38479.811550,20418.295618
std,18563.411961,9993.013630
min,762.959000,406.210000
25%,25548.938658,13338.012962
50%,38306.374333,20217.558063
75%,51149.360279,27241.726091
max,125170.657315,60683.028105


In [72]:
feat_info_icd10.sample(2)

,icd10_cm_code,elective_rate,ln_mean_icd10,ln_median_icd10
4115,M130,0.168792,87857.529414,17317.080661
7173,S42035A,0.169244,35219.852022,23449.849240


In [71]:
feat_info_icd10.to_csv(Path(setup.output_path).joinpath('02_feat_icd10_based_info.csv'), index=False)

## ICD10 and CCSR Code-Based Costs

#### Function to 

In [76]:
icd_cost_param = feat_info_icd10[['icd10_cm_code', 'ln_mean_icd10', 'ln_median_icd10']] \
    .merge(icd10_map[['icd10_cm_code', 'ccsr_code']], on='icd10_cm_code') \
    .merge(feat_info_ccsr[['ccsr_code', 'ln_mean_ccsr', 'ln_median_ccsr']], on ='ccsr_code')

In [78]:
icd_cost_param.sample(3)

,icd10_cm_code,ln_mean_icd10,ln_median_icd10,ccsr_code,ln_mean_ccsr,ln_median_ccsr
2592,I63413,48549.333552,28783.294332,CIR020,39043.611180,24114.542217
980,D483,21769.466874,12449.647441,NEO072,33126.620018,17001.277019
573,C720,2165.022062,25102.585331,NEO049,39210.741485,14317.671898


In [80]:
grouped = visits_df.groupby('icd10_cm_code')

In [101]:
for n, g in grouped:
    this_params = icd_cost_param[icd_cost_param['icd10_cm_code'] == n].iloc[0]
    print(this_params)
    print(len(g))
    this_ln_icd10 = get_log_norm_dist(mean=this_params.ln_mean_icd10,
                                    median=this_params.ln_median_icd10)
    cost_ser_icd10 = pd.Series(this_ln_icd10.rvs(size=len(g), random_state=135),
                         index=g.index, name='icd10_cm_code__dollars')
    this_ln_ccsr = get_log_norm_dist(mean=this_params.ln_mean_ccsr,
                                    median=this_params.ln_median_ccsr)
    cost_ser_ccsr = pd.Series(this_ln_ccsr.rvs(size=len(g), random_state=656),
                         index=g.index, name='ccsr_code__dollars')
    tot_cost = cost_ser_icd10 + cost_ser_ccsr
    break

icd10_cm_code             A0100
ln_mean_icd10      26976.262162
ln_median_icd10     9042.474927
ccsr_code                DIG001
ln_mean_ccsr       34911.439698
ln_median_ccsr     21795.953405
Name: 0, dtype: object
3


In [102]:
this_params.ln_mean_icd10

np.float64(26976.26216156913)

In [103]:
cost_ser_icd10.head(10)

46397      1393.482944
256949    52087.599171
456432    15285.123654
Name: icd10_cm_code__dollars, dtype: float64

In [104]:
cost_ser_ccsr.head(10)

46397      6784.616026
256949    11422.935967
456432    31265.903674
Name: ccsr_code__dollars, dtype: float64

In [105]:
tot_cost.head(10)

46397      8178.098971
256949    63510.535137
456432    46551.027328
dtype: float64

In [ ]:
def get_log_norm_dist(mean=cost_mean, median=cost_median):